# build_knowledge_colab.py

In [ ]:
%%writefile build_knowledge_colab.py
import os
import json
import tempfile
from openai import OpenAI


def load_json(path):
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    return {}


def save_json(path, data):
    folder = os.path.dirname(path)
    if folder:
        os.makedirs(folder, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


# Inventory/cost fields in the hospital formulary that carry zero clinical value
_FORMULARY_SKIP = {
    "Seq", "Code", "StoreBin", "Quantity", "UOM",
    "ItemCost", "Amount", "Count", "Vendor", "Unnamed: 10", "Manufacturing",
    "Generic Drug",
}
# Truncate long narrative fields so each entry stays under ~600 chars
_FORMULARY_TRUNCATE = {
    "Indications": 200,
    "Dosage (mg/kg/day)": 150,
    "Contraindications": 250,
}


def _format_formulary(obj):
    """Compact single-block text for a BKN formulary drug entry."""
    name  = obj.get("Generic (ENG)") or obj.get("Itemname") or "Unknown"
    brand = obj.get("Itemname", "")
    group = obj.get(" Drug Group") or obj.get("Drug Group", "")
    allergen = str(obj.get("Allergen", "")).strip()

    header = f"Drug: {name}"
    if brand and brand.lower() != str(name).lower():
        header += f" ({brand})"
    if group:
        header += f" | Group: {group}"

    parts = [header]

    if allergen and allergen not in ("nan", "NaN", ""):
        if group and allergen.startswith(group + ","):
            allergen = allergen[len(group) + 1:].strip()
        parts.append(f"Allergen: {allergen}")

    for field, limit in _FORMULARY_TRUNCATE.items():
        val = str(obj.get(field, "")).strip()
        if val and val not in ("nan", "NaN", "-"):
            truncated = val[:limit] + ("..." if len(val) > limit else "")
            label = "Dosage" if "Dosage" in field else field
            parts.append(f"{label}: {truncated}")

    return "\n".join(parts)


def _format_knowledge(obj):
    """Compact readable text for taxonomy/allergy KB entries."""
    parts = []

    title = obj.get("title") or obj.get("label") or ""
    if title:
        parts.append(f"[{title}]")

    allergen_class = obj.get("allergen_class", "")
    allergen_examples = obj.get("allergen_examples", [])
    risk = obj.get("risk_level", "")
    if allergen_class:
        ex = ", ".join(allergen_examples[:4]) if allergen_examples else ""
        line = f"Allergen class: {allergen_class}"
        if ex:
            line += f" (e.g. {ex})"
        if risk:
            line += f" — Risk: {risk}"
        parts.append(line)

    text = obj.get("text", "")
    if text:
        parts.append(text)

    criteria = obj.get("criteria", [])
    if criteria:
        parts.append("Detection criteria: " + "; ".join(criteria))

    action = obj.get("action", "")
    if action:
        parts.append(f"Action: {action}")

    return "\n".join(parts)


def _is_formulary(first_obj):
    """Detect if this JSONL is a drug formulary (has Itemname + Allergen fields)."""
    return "Itemname" in first_obj and "Allergen" in first_obj


def _jsonl_to_txt(filepath):
    """Convert .jsonl to lean plain text for OpenAI upload.

    Formulary files: strips inventory fields, truncates narrative fields (~80% savings).
    Knowledge/taxonomy files: formats as readable sections (~35% savings).
    """
    base = os.path.splitext(os.path.basename(filepath))[0]
    tmp_path = os.path.join(tempfile.gettempdir(), f"{base}.txt")

    with open(filepath, "r", encoding="utf-8") as in_f:
        raw_lines = [l.strip() for l in in_f if l.strip()]

    if not raw_lines:
        open(tmp_path, "w").close()
        return tmp_path

    try:
        first_obj = json.loads(raw_lines[0])
        formulary = _is_formulary(first_obj)
    except json.JSONDecodeError:
        formulary = False

    formatter = _format_formulary if formulary else _format_knowledge

    with open(tmp_path, "w", encoding="utf-8") as out_f:
        for lineno, line in enumerate(raw_lines, 1):
            try:
                obj = json.loads(line)
                out_f.write(formatter(obj) + "\n\n")
            except json.JSONDecodeError as e:
                print(f"Warning: skipping invalid JSON on line {lineno}: {e}")

    return tmp_path


def build_knowledge(
    api_key,
    knowledge_files,
    vector_store_name,
    cache_path,
    vs_id_path=None,
    force_new_vector_store=False,
):
    client = OpenAI(api_key=api_key)

    cache = load_json(cache_path)
    files_cache = cache.get("files", {}) if isinstance(cache.get("files"), dict) else {}
    vector_store_id = None if force_new_vector_store else cache.get("vector_store", {}).get("id")

    # Verify cached vector store still exists on OpenAI
    if vector_store_id:
        try:
            client.vector_stores.retrieve(vector_store_id)
            print(f"Reusing vector store: {vector_store_id}")
        except Exception:
            print(f"Vector store {vector_store_id} expired — will create a new one.")
            vector_store_id = None
            files_cache = {}

    # Stage 1: resolve upload paths (.jsonl → .txt)
    files_to_process = []
    temp_files = []

    for filepath in knowledge_files:
        if not os.path.exists(filepath):
            raise FileNotFoundError(f"File not found: {filepath}")
        if filepath.lower().endswith(".jsonl"):
            print(f"Converting {os.path.basename(filepath)} → .txt ...")
            tmp = _jsonl_to_txt(filepath)
            cache_key = os.path.splitext(os.path.basename(filepath))[0] + ".txt"
            temp_files.append(tmp)
            files_to_process.append((tmp, cache_key))
        else:
            files_to_process.append((filepath, os.path.basename(filepath)))

    # Stage 2: upload files (skip cached)
    uploaded_file_ids = []
    for upload_path, cache_key in files_to_process:
        if cache_key in files_cache and not force_new_vector_store:
            file_id = files_cache[cache_key]
            print(f"Reusing cached file: {cache_key} -> {file_id}")
        else:
            print(f"Uploading: {cache_key} ...")
            with open(upload_path, "rb") as f:
                uploaded = client.files.create(file=(cache_key, f.read()), purpose="assistants")
            file_id = uploaded.id
            files_cache[cache_key] = file_id
            print(f"Uploaded: {cache_key} -> {file_id}")
        uploaded_file_ids.append(file_id)

    # Stage 3: create or reuse vector store
    if not vector_store_id:
        print(f"Creating vector store '{vector_store_name}' ...")
        vs = client.vector_stores.create(name=vector_store_name)
        vector_store_id = vs.id
        print(f"Created vector store: {vector_store_id}")

    # Stage 4: attach files and wait for indexing
    print("Attaching files and waiting for indexing ...")
    batch = client.vector_stores.file_batches.create_and_poll(
        vector_store_id=vector_store_id,
        file_ids=uploaded_file_ids,
    )
    print(f"Batch status: {batch.status}")
    if getattr(batch, "file_counts", None):
        print(f"File counts: {batch.file_counts}")

    if batch.status != "completed":
        raise RuntimeError(f"Indexing did not complete: status={batch.status}")

    # Persist cache
    save_json(cache_path, {
        "vector_store": {"id": vector_store_id, "name": vector_store_name},
        "files": files_cache,
    })

    # Save vector store ID to dedicated file for easy reuse
    id_path = vs_id_path or os.path.join(os.path.dirname(cache_path), "vector_store_id.txt")
    os.makedirs(os.path.dirname(id_path), exist_ok=True)
    with open(id_path, "w") as f:
        f.write(vector_store_id)
    print(f"\nVector store ID saved to: {id_path}")

    # Cleanup temp files
    for tmp in temp_files:
        try:
            os.remove(tmp)
        except OSError:
            pass

    print(f"\nDONE. VECTOR_STORE_ID = {vector_store_id}")
    return {"vector_store_id": vector_store_id, "batch_status": batch.status}


# run_inference_colab.py

In [ ]:
%%writefile run_inference_colab.py
import os
import json
import time
import traceback
from typing import List, Dict, Any

import pandas as pd
from pydantic import BaseModel
from openai import OpenAI, APIConnectionError, APITimeoutError, RateLimitError, APIError


class MediCheckOutput(BaseModel):
    order_id: str
    has_medication_error: bool
    error_categories: List[str]
    error_details: List[str]
    implicated_drugs: List[str]


def load_json(path, default=None):
    if default is None:
        default = {}
    if path and os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    return default


def save_json(path, data):
    folder = os.path.dirname(path)
    if folder:
        os.makedirs(folder, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


def normalize_order_id(value):
    if value is None:
        return "UNKNOWN"
    return str(value).strip()


def dedupe_preserve_order(items):
    seen, result = set(), []
    for item in items:
        if item is None:
            continue
        item = str(item).strip()
        if item and item not in seen:
            seen.add(item)
            result.append(item)
    return result


def clean_dataframe(df):
    df = df.dropna(how="all")
    df = df.apply(lambda x: x.str.strip() if x.dtype == "object" else x)
    df = df.where(pd.notnull(df), None)
    df = df.drop_duplicates()
    if "Drug" in df.columns:
        df = df[~df["Drug"].astype(str).str.strip().isin(["", "-", "None", "nan"])]
    return df


INSTRUCTIONS = """
You are a clinical AI system for classifying prescription medication errors.
- Use ONLY the knowledge excerpts and prescription data provided.
- Every detected error MUST be supported by explicit evidence from the knowledge.
- If evidence is missing or insufficient, set has_medication_error to false.
- Multi-label classification is allowed when independently supported.
- Precision over recall: it is better to miss an error than to falsely label one.
- Return JSON only. No extra text.
""".strip()


def get_json_schema():
    return {
        "type": "object",
        "additionalProperties": False,
        "properties": {
            "order_id": {"type": "string"},
            "has_medication_error": {"type": "boolean"},
            "error_categories": {"type": "array", "items": {"type": "string"}},
            "error_details": {"type": "array", "items": {"type": "string"}},
            "implicated_drugs": {"type": "array", "items": {"type": "string"}},
        },
        "required": ["order_id", "has_medication_error", "error_categories", "error_details", "implicated_drugs"],
    }


def build_user_prompt(order_id, records):
    compact = [{"Drug": r.get("Drug"), "Dose": r.get("Dose"), "Frequency": r.get("Frequency")} for r in records]
    return (
        f"Order ID: {order_id}\n"
        f"Prescription: {json.dumps(compact, ensure_ascii=False, separators=(',', ':'))}\n"
        f"Classify medication errors. Return JSON only."
    )


def build_kb_query(records):
    drugs = dedupe_preserve_order([
        str(r.get("Drug", "")).strip()
        for r in records
        if r.get("Drug") and str(r.get("Drug")).strip() not in ["-", "None", "nan"]
    ])
    return f"medication error classification {' '.join(drugs[:5])}".strip()


def retrieve_kb_snippets(client, vector_store_id, query, max_results=5):
    result = client.vector_stores.search(
        vector_store_id=vector_store_id,
        query=query,
        max_num_results=max_results,
    )
    snippets = []
    for item in getattr(result, "data", []):
        parts = [getattr(c, "text", None) for c in getattr(item, "content", []) if getattr(c, "text", None)]
        joined = "\n".join(parts).strip()
        if joined:
            snippets.append(joined)
    return "\n\n---\n\n".join(snippets)


def call_model(client, model, vector_store_id, user_prompt, records, max_num_results=5, max_retries=3):
    kb_query = build_kb_query(records)
    last_error = None

    for attempt in range(1, max_retries + 1):
        try:
            kb_snippets = retrieve_kb_snippets(client, vector_store_id, kb_query, max_num_results)
            if not kb_snippets.strip():
                raise ValueError("No knowledge snippets retrieved from vector store.")

            response = client.responses.create(
                model=model,
                instructions=INSTRUCTIONS,
                input=f"KNOWLEDGE:\n{kb_snippets}\n\nCASE:\n{user_prompt}",
                text={"format": {"type": "json_schema", "name": "medicheck_output", "strict": True, "schema": get_json_schema()}},
            )
            raw_text = getattr(response, "output_text", None)
            if not raw_text or not raw_text.strip():
                raise ValueError("Model returned empty output.")
            return json.loads(raw_text)

        except (APIConnectionError, APITimeoutError, RateLimitError, APIError) as e:
            last_error = e
            time.sleep(min(2 ** attempt, 10))
        except Exception as e:
            last_error = e
            if attempt < max_retries:
                time.sleep(1)

    raise last_error


def postprocess(parsed, order_id):
    parsed["order_id"] = normalize_order_id(parsed.get("order_id", order_id))
    for key in ["error_categories", "error_details", "implicated_drugs"]:
        parsed[key] = dedupe_preserve_order(parsed.get(key, []))
    if not parsed.get("has_medication_error", False):
        parsed["error_categories"] = []
        parsed["error_details"] = []
        parsed["implicated_drugs"] = []
    return MediCheckOutput(**parsed).model_dump()


def run_inference(
    api_key,
    excel_file,
    group_column,
    output_dir,
    vector_store_id=None,     # ← pass directly from VECTOR_STORE_ID variable
    cache_path=None,          # ← fallback if vector_store_id not provided
    model="gpt-4o-mini",
    sleep_sec=1.0,
    autosave_every=10,
    max_num_results=5,
    debug=False,
):
    # Resolve vector store ID
    if not vector_store_id:
        cache = load_json(cache_path)
        vector_store_id = cache.get("vector_store", {}).get("id", "")
    if not vector_store_id:
        raise ValueError("vector_store_id not found. Run the build cell first.")

    print(f"Using vector store: {vector_store_id}")
    client = OpenAI(api_key=api_key)
    os.makedirs(output_dir, exist_ok=True)

    partial_file = os.path.join(output_dir, "partial_results.json")
    failed_file  = os.path.join(output_dir, "failed_cases.json")
    final_file   = os.path.join(output_dir, "medicheck_results.json")

    results      = load_json(partial_file, {})
    failed_cases = load_json(failed_file, [])

    if not os.path.exists(excel_file):
        raise FileNotFoundError(f"Input file not found: {excel_file}")

    if excel_file.lower().endswith(".json"):
        with open(excel_file, "r", encoding="utf-8") as f:
            df = pd.DataFrame(json.load(f))
    else:
        df = pd.read_excel(excel_file)

    df = clean_dataframe(df)
    if group_column not in df.columns:
        raise ValueError(f"Column '{group_column}' not found. Available: {list(df.columns)}")

    grouped = list(df.groupby(group_column, dropna=False))
    total = len(grouped)
    remaining = [(normalize_order_id(oid), gdf) for oid, gdf in grouped if normalize_order_id(oid) not in results]
    print(f"Already processed: {total - len(remaining)} / {total}")
    print(f"Remaining: {len(remaining)}")

    for idx, (order_id, group_df) in enumerate(remaining, start=1):
        current = (total - len(remaining)) + idx
        print(f"[{current}/{total}] {order_id}", end=" ... ")
        try:
            records = group_df.to_dict(orient="records")
            prompt  = build_user_prompt(order_id, records)
            parsed  = call_model(client, model, vector_store_id, prompt, records, max_num_results)
            results[order_id] = postprocess(parsed, order_id)
            print("OK")
        except Exception as e:
            failed_cases.append({"order_id": order_id, "error": str(e), "traceback": traceback.format_exc()})
            print(f"FAILED: {e}")

        time.sleep(sleep_sec)

        if current % autosave_every == 0:
            save_json(partial_file, results)
            save_json(failed_file, failed_cases)
            print(f"  → Autosaved at {current} cases")

    save_json(final_file, results)
    save_json(failed_file, failed_cases)

    print(f"\nDONE — processed: {len(results)}, failed: {len(failed_cases)}")
    return {"results_file": final_file, "processed": len(results), "failed": len(failed_cases)}


# Variations

In [ ]:
import os
from google.colab import drive, userdata

drive.mount('/content/drive')

# ── API & model ────────────────────────────────────────────────────────────────
OPENAI_API_KEY = userdata.get("GPTKEY")
MODEL = "gpt-4o-mini"

# ── Paths (all on Google Drive so they persist across sessions) ────────────────
KB_DIR       = "/content/drive/MyDrive/thesis/KB"
CACHE_DIR    = "/content/drive/MyDrive/thesis/Result/medicheck_cache"
CACHE_PATH   = f"{CACHE_DIR}/knowledge_cache.json"
VS_ID_PATH   = f"{CACHE_DIR}/vector_store_id.txt"   # ← persistent ID file
OUTPUT_DIR   = "/content/drive/MyDrive/thesis/Result/thesismedicheck_output"
EXCEL_FILE   = "/content/drive/MyDrive/thesis/RT_COMMON_904_test_clean_blinded_first_sheet.json"
GROUP_COLUMN = "ID"

# ── Knowledge base files ───────────────────────────────────────────────────────
KNOWLEDGE_FILES = [
    f"{KB_DIR}/medication_error_kb_chunked.jsonl",
    f"{KB_DIR}/allergy_rules.jsonl",
    f"{KB_DIR}/BKN_Med_List.jsonl",
    f"{KB_DIR}/index-color-2021-draft-change-10-2022.pdf",
    f"{KB_DIR}/MedFacts - Pocket Guide of Drug Interaction.pdf",
    f"{KB_DIR}/DrugInteractionsChapter.pdf",
    f"{KB_DIR}/คู่มือการใช้ยาที่มีความเสี่ยงสูง High Alert Drug ปรับปรุง 2564 HAD_2109.pdf",
    f"{KB_DIR}/การจัดการยาความเสี่ยงสูง.pdf",
]
VECTOR_STORE_NAME = "MediCheck Medication Error KB"

# ── Load saved vector store ID (if already built) ─────────────────────────────
VECTOR_STORE_ID = ""
if os.path.exists(VS_ID_PATH):
    with open(VS_ID_PATH) as f:
        VECTOR_STORE_ID = f.read().strip()
    print(f"✅ Loaded VECTOR_STORE_ID: {VECTOR_STORE_ID}")
    print("   → Skip build cell. Go straight to inference.")
else:
    print("⚠️  No saved VECTOR_STORE_ID found.")
    print("   → Run the 'Build knowledge base' cell below (once only).")


# Create knowledge base

In [ ]:
import importlib, build_knowledge_colab
importlib.reload(build_knowledge_colab)
from build_knowledge_colab import build_knowledge

# ── Verify all KB files exist ──────────────────────────────────────────────────
all_ok = True
for f in KNOWLEDGE_FILES:
    status = "✅ OK" if os.path.exists(f) else "❌ NOT FOUND"
    print(f"{status}  {os.path.basename(f)}")
    if "NOT FOUND" in status:
        all_ok = False

if not all_ok:
    raise FileNotFoundError("Some knowledge files are missing. Check paths above.")

# ── Build (runs once — reuses everything if force_new_vector_store=False) ──────
# Set force_new_vector_store=True ONLY when KB files have changed
build_result = build_knowledge(
    api_key=OPENAI_API_KEY,
    knowledge_files=KNOWLEDGE_FILES,
    vector_store_name=VECTOR_STORE_NAME,
    cache_path=CACHE_PATH,
    vs_id_path=VS_ID_PATH,
    force_new_vector_store=False,   # ← keep False; set True only to rebuild KB
)

VECTOR_STORE_ID = build_result["vector_store_id"]
print(f"\nVECTOR_STORE_ID = {VECTOR_STORE_ID}")


# Run Inference

In [ ]:
import importlib, run_inference_colab
importlib.reload(run_inference_colab)
from run_inference_colab import run_inference

if not VECTOR_STORE_ID:
    raise ValueError("VECTOR_STORE_ID is empty. Run config cell — if ID loads, skip build. Otherwise run build cell first.")

run_result = run_inference(
    api_key=OPENAI_API_KEY,
    excel_file=EXCEL_FILE,
    group_column=GROUP_COLUMN,
    vector_store_id=VECTOR_STORE_ID,   # ← passed directly, no cache lookup needed
    output_dir=OUTPUT_DIR,
    model=MODEL,
    sleep_sec=1.0,
    autosave_every=10,
    max_num_results=5,
    debug=False,
)

run_result


In [ ]:
# recheck vector
from openai import OpenAI
import json

client = OpenAI(api_key=OPENAI_API_KEY)

cache = json.load(open(CACHE_PATH))
vs_id = cache["vector_store"]["id"]

vs = client.vector_stores.retrieve(vs_id)

print("status:", vs.status)
print("file_counts:", vs.file_counts)

status: completed
file_counts: FileCounts(cancelled=0, completed=2, failed=0, in_progress=0, total=2)


In [ ]:
import importlib
import run_inference_colab
importlib.reload(run_inference_colab)

from run_inference_colab import run_inference

run_result = run_inference(
    api_key=OPENAI_API_KEY,
    excel_file=EXCEL_FILE,
    group_column=GROUP_COLUMN,
    cache_path=CACHE_PATH,
    output_dir=OUTPUT_DIR,
    model=MODEL,
    sleep_sec=1.0,
    autosave_every=10,
    max_num_results=8,
    debug=DEBUG,
)

run_result

Already processed: 40
Remaining: 8 / 48

Processing 41/48 -> Order d36d882e3c38
----- KB QUERY -----
medication error classification taxonomy relevant drugs parecoxib sodium; ondansetron, ondansetron hydrochloride
----- KB SNIPPETS -----
{"id": "med_error_definition", "type": "definition", "topic": "medication_error_definition", "title": "Medication Error Definition", "text": "A failure in the treatment process that leads to, or has the potential to lead to, harm to the patient.", "keywords": ["medication error", "definition", "harm", "treatment process"]}
{"id": "prescribing_fault_definition", "type": "definition", "topic": "medication_error_definition", "title": "Prescribing Fault", "text": "Failure in decision-making process of prescribing", "keywords": ["prescribing fault", "decision-making", "prescribing process"]}
{"id": "prescription_error_definition", "type": "definition", "topic": "medication_error_definition", "title": "Prescription Error", "text": "Failure in prescription wr

{'results_file': '/content/drive/MyDrive/thesis/Result/thesismedicheck_output/medicheck_results.json',
 'failed_file': '/content/drive/MyDrive/thesis/Result/thesismedicheck_output/failed_cases.json',
 'debug_file': '/content/drive/MyDrive/thesis/Result/thesismedicheck_output/debug_log.json',
 'processed': 48,
 'failed': 0}

# Result

In [ ]:
import json
import os

result_file = f"/content/drive/MyDrive/medicheck_output/partial_results.json"
failed_file = f"/content/drive/MyDrive/medicheck_output/failed_cases.json"
debug_file = f"/content/drive/MyDrive/medicheck_output/debug_log.json"

print("result_file exists:", os.path.exists(result_file))
print("failed_file exists:", os.path.exists(failed_file))
print("debug_file exists:", os.path.exists(debug_file))

with open(result_file, "r", encoding="utf-8") as f:
    results = json.load(f)

with open(failed_file, "r", encoding="utf-8") as f:
    failed = json.load(f)

print("success cases =", len(results))
print("failed cases =", len(failed))

result_file exists: True
failed_file exists: True
debug_file exists: True
success cases = 0
failed cases = 30


In [ ]:
import pandas as pd

rows = []
for order_id, item in results.items():
    rows.append({
        "order_id": item.get("order_id", order_id),
        "has_medication_error": item.get("has_medication_error"),
        "error_categories": " | ".join(item.get("error_categories", [])),
        "error_details": " | ".join(item.get("error_details", [])),
        "implicated_drugs": " | ".join(item.get("implicated_drugs", [])),
    })

df_result = pd.DataFrame(rows)
df_result.head()

""


In [ ]:
import json

with open(f"/content/drive/MyDrive/medicheck_output/debug_log.json", "r") as f:
    logs = json.load(f)

logs[0]

{'order_id': '0280d683793d',
 'error_type': 'ValueError',
 'error': 'Model returned empty text output.',
 'prompt_preview': '\nReview the following prescription order.\n\nOrder ID: 0280d683793d\n\nPrescription data:\n[\n  {\n    "ID": "0280d683793d",\n    "Drug": "diazepam",\n    "Dose": "1 Capsule",\n    "Frequency": "2 Time Daily Morning and Evening"\n  },\n  {\n    "ID": "0280d683793d",\n    "Drug": "ondansetron",\n    "Dose": "1 Tablet",\n    "Frequency": "3 Times Daily Before Breakfast Lunch and "\n  },\n  {\n    "ID": "0280d683793d",\n    "Drug": "hyoscine-n-butyl bromide, hyoscine-n-",\n    "Dose": "1 Tablet",\n    "Frequency": "Every 6 Hours If Needed For Abdominal "\n  },\n  {\n    "ID": "0280d683793d",\n    "Drug": "-",\n    "Dose": "- -",\n    "Frequency": "Every 2-3 Hours When have Symptom"\n  }\n]\n\nTasks:\n1. Decide whether this prescription contains a medication error.\n2. Apply multi-label classification if more than one issue is present.\n3. Classify the issue using t

# inspect failures

In [ ]:
import json

with open(f"{OUTPUT_DIR}/failed_cases.json", "r", encoding="utf-8") as f:
    failed = json.load(f)

print("failed count =", len(failed))
failed[:3]

failed count = 0


[]